In [1]:
import os
import gc
import math
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [2]:
import os
os.makedirs("/kaggle/tmp", exist_ok=True)

In [3]:
SAVE_PATH = "/kaggle/working/checkpoints/"

os.makedirs(SAVE_PATH, exist_ok=True)

In [4]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if "LOOPSEA_10" in f:
            print(os.path.join(root, f))

/kaggle/input/datasets/chetannarware01/final-project-data/Final_Project_Data/LOOPSEA_10percent_missing.csv


In [5]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/working/__notebook__.ipynb


In [6]:
# files = {
#     10: "/kaggle/input/datasets/chetannarware01/final-project-data/Final_Project_Data/LOOPSEA_10percent_missing.csv",
#     20: "/kaggle/input/datasets/chetannarware01/final-project-data/Final_Project_Data/LOOPSEA_20percent_missing.csv",
#     40: "/kaggle/input/datasets/chetannarware01/final-project-data/Final_Project_Data/LOOPSEA_40percent_missing.csv",
#     80: "/kaggle/input/datasets/chetannarware01/final-project-data/Final_Project_Data/LOOPSEA_80percent_missing.csv",
#      0: "/kaggle/input/datasets/chetannarware01/final-project-data/Final_Project_Data/LOOPSEA.csv"
# }
files = {
    10: "/kaggle/input/datasets/chetannarware01/pims-bay/missing_data/pemsbay_10percent_missing.csv",
    20: "/kaggle/input/datasets/chetannarware01/pims-bay/missing_data/pemsbay_20percent_missing.csv",
    40: "/kaggle/input/datasets/chetannarware01/pims-bay/missing_data/pemsbay_40percent_missing.csv",
    80: "/kaggle/input/datasets/chetannarware01/pims-bay/missing_data/pemsbay_80percent_missing.csv",
     0: "/kaggle/input/datasets/chetannarware01/pims-bay/missing_data/pemsbay_0percent_missing.csv"
}
print("Files configured:", list(files.keys()))

Files configured: [10, 20, 40, 80, 0]


In [7]:
def create_sequences(data, mask, seq_len=10):
    X, M, Y = [], [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        M.append(mask[i:i+seq_len])
        Y.append(data[i+seq_len])
    return np.array(X), np.array(M), np.array(Y)

In [8]:
class TrafficDataset(Dataset):
    def __init__(self, X, M, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.M = torch.tensor(M, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.M[idx], self.Y[idx]

In [9]:
class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.conv = nn.Conv1d(
            input_dim + hidden_dim + 1,  # +1 for mask channel in gates
            4 * hidden_dim,
            kernel_size=3,
            padding=1
        )
        self.norm    = nn.GroupNorm(4, hidden_dim)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x, h, c, m=None):
        if m is None:
            m = torch.ones(x.shape[0], 1, x.shape[2], device=x.device)
        combined = torch.cat([x, h, m], dim=1)
        i, f, o, g = torch.chunk(self.conv(combined), 4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)
        c = f * c + i * g
        h = o * torch.tanh(c)
        h = self.norm(h)
        h = self.dropout(h)
        return h, c

In [10]:
class IConvBiLSTM(nn.Module):
    """
    IConvBiLSTM: Imputation ConvBiLSTM
    Extends paper's BDLSTM-I with:
      1. Conv1d gates instead of matrix multiply (spatial context)
      2. Separate fwd/bwd imputation units with wider kernel
      3. Mask explicitly passed to all LSTM gates (Eq. 13-16)
      4. Residual connections into layer 2
      5. Temporal attention over all T steps for prediction
      6. GroupNorm for training stability
    """
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.hidden_dim = hidden_dim

        # Separate imputation units — wider kernel for spatial context
        self.impute_fwd = nn.Sequential(
            nn.Conv1d(2 * hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(hidden_dim, 1, kernel_size=3, padding=1)
        )
        self.impute_bwd = nn.Sequential(
            nn.Conv1d(2 * hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(hidden_dim, 1, kernel_size=3, padding=1)
        )

        # Layer 1: BDLSTM-I (imputation + bidirectional)
        # input_dim=2 → x_prime(1ch) + mask(1ch)
        self.convlstm1_fwd = ConvLSTMCell(input_dim=2, hidden_dim=hidden_dim)
        self.convlstm1_bwd = ConvLSTMCell(input_dim=2, hidden_dim=hidden_dim)

        # Layer 2: BDLSTM (no imputation, takes concatenated layer1 output)
        # input_dim=2*hidden → fwd(hidden) + bwd(hidden) concatenated
        self.convlstm2_fwd = ConvLSTMCell(input_dim=2 * hidden_dim, hidden_dim=hidden_dim)
        self.convlstm2_bwd = ConvLSTMCell(input_dim=2 * hidden_dim, hidden_dim=hidden_dim)

        # Residual projection for layer1→layer2 connection
        self.residual_proj = nn.Conv1d(2 * hidden_dim, 2 * hidden_dim, kernel_size=1)

        # Temporal attention: scores each of T steps
        self.temporal_attn = nn.Conv1d(hidden_dim, 1, kernel_size=1)

        # Output head
        self.output_head = nn.Conv1d(hidden_dim, 1, kernel_size=1)

    def forward(self, x, mask):
        B, T, F = x.shape

        # Layer 1 states
        h1f = torch.zeros(B, self.hidden_dim, F, device=x.device)
        c1f = torch.zeros(B, self.hidden_dim, F, device=x.device)
        h1b = torch.zeros(B, self.hidden_dim, F, device=x.device)
        c1b = torch.zeros(B, self.hidden_dim, F, device=x.device)

        # Layer 2 states
        h2f = torch.zeros(B, self.hidden_dim, F, device=x.device)
        c2f = torch.zeros(B, self.hidden_dim, F, device=x.device)
        h2b = torch.zeros(B, self.hidden_dim, F, device=x.device)
        c2b = torch.zeros(B, self.hidden_dim, F, device=x.device)

        fwd_h1_steps  = []
        fwd_imp_steps = []

        # === LAYER 1 FORWARD (Eq. 10-18) ===
        for t in range(T):
            xt = x[:, t, :].unsqueeze(1)      # (B, 1, F)
            mt = mask[:, t, :].unsqueeze(1)   # (B, 1, F)

            # Impute from own preceding h and c (Eq. 10)
            x_tilde_fwd = self.impute_fwd(torch.cat([h1f, c1f], dim=1))
            # Mask fusion (Eq. 12)
            x_prime     = mt * xt + (1.0 - mt) * x_tilde_fwd
            x_with_mask = torch.cat([x_prime, mt], dim=1)         # (B, 2, F)
            # Gate update with mask (Eq. 13-18)
            h1f, c1f = self.convlstm1_fwd(x_with_mask, h1f, c1f, mt)
            fwd_h1_steps.append(h1f)
            fwd_imp_steps.append(x_tilde_fwd.squeeze(1))

        # === LAYER 1 BACKWARD (Eq. 10-18, reversed) ===
        bwd_h1_steps  = [None] * T
        bwd_imp_steps = [None] * T
        for t in reversed(range(T)):
            xt = x[:, t, :].unsqueeze(1)
            mt = mask[:, t, :].unsqueeze(1)

            x_tilde_bwd = self.impute_bwd(torch.cat([h1b, c1b], dim=1))
            x_prime     = mt * xt + (1.0 - mt) * x_tilde_bwd
            x_with_mask = torch.cat([x_prime, mt], dim=1)
            h1b, c1b    = self.convlstm1_bwd(x_with_mask, h1b, c1b, mt)
            bwd_h1_steps[t]  = h1b
            bwd_imp_steps[t] = x_tilde_bwd.squeeze(1)

        # === LAYER 2 FORWARD with residual ===
        l2_fwd_steps = []
        for t in range(T):
            mt     = mask[:, t, :].unsqueeze(1)
            y1_cat = torch.cat([fwd_h1_steps[t], bwd_h1_steps[t]], dim=1)
            y1     = y1_cat + self.residual_proj(y1_cat)           # residual
            h2f, c2f = self.convlstm2_fwd(y1, h2f, c2f, mt)       # mask to gates
            l2_fwd_steps.append(h2f)

        # === LAYER 2 BACKWARD with residual ===
        l2_bwd_steps = [None] * T
        for t in reversed(range(T)):
            mt     = mask[:, t, :].unsqueeze(1)
            y1_cat = torch.cat([fwd_h1_steps[t], bwd_h1_steps[t]], dim=1)
            y1     = y1_cat + self.residual_proj(y1_cat)           # residual
            h2b, c2b = self.convlstm2_bwd(y1, h2b, c2b, mt)       # mask to gates
            l2_bwd_steps[t] = h2b

        # === TEMPORAL ATTENTION over all T steps (Eq. 20 extended) ===
        l2_avg = [0.5 * (l2_fwd_steps[t] + l2_bwd_steps[t])
                  for t in range(T)]                               # list of (B, hidden, F)

        attn_scores  = torch.stack(
            [self.temporal_attn(h) for h in l2_avg], dim=1
        )                                                          # (B, T, 1, F)
        attn_weights = torch.softmax(attn_scores, dim=1)          # (B, T, 1, F)
        l2_stack     = torch.stack(l2_avg, dim=1)                 # (B, T, hidden, F)
        y2_attn      = (attn_weights * l2_stack).sum(dim=1)       # (B, hidden, F)

        pred = self.output_head(y2_attn).squeeze(1)               # (B, F)

        imp_fwd = torch.stack(fwd_imp_steps, dim=1)               # (B, T, F)
        imp_bwd = torch.stack(bwd_imp_steps, dim=1)               # (B, T, F)

        return pred, imp_fwd, imp_bwd


In [11]:
def compute_loss(pred, target, imp_fwd, imp_bwd, x_input, mask, lam=0.1):
    """
    Eq. 22 from Cui et al. (2020):
    L = Loss(y_T - x_{T+1}) + λ * Σ_t Σ_{observed} 0.5*(|x-x̃_fwd| + |x-x̃_bwd|)
    """
    pred_loss = torch.mean(torch.abs(pred - target)) + \
                0.1 * torch.mean((pred - target) ** 2)

    # Average fwd and bwd imputation errors on observed positions only
    imp_error = 0.5 * (
        torch.abs(imp_fwd - x_input) + torch.abs(imp_bwd - x_input)
    ) * mask
    imp_loss = imp_error.sum() / (mask.sum() + 1e-8)

    return pred_loss + lam * imp_loss

In [12]:
def hybrid_loss(pred, target):
    mse = torch.mean((pred - target) ** 2)
    mae = torch.mean(torch.abs(pred - target))
    return 0.5 * mse + mae

In [13]:
from tqdm.auto import tqdm

def train_model(model, train_loader, val_loader, percent,
                max_epochs=70, patience=15):             # 80 → 70
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-5
    )
    best_loss = float("inf")
    wait      = 0
    best_path = f"/kaggle/tmp/best_{percent}.pt"

    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{max_epochs}",
                    leave=False, mininterval=2.0)
        for x, m, y in pbar:
            x, m, y = (x.to(device, non_blocking=True),
                       m.to(device, non_blocking=True),
                       y.to(device, non_blocking=True))
            optimizer.zero_grad()
            pred, imp_fwd, imp_bwd = model(x, m)
            loss = compute_loss(pred, y, imp_fwd, imp_bwd, x, m, lam=0.1)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x, m, y in val_loader:
                x, m, y = (x.to(device, non_blocking=True),
                           m.to(device, non_blocking=True),
                           y.to(device, non_blocking=True))
                pred, _, _ = model(x, m)
                val_loss += (torch.mean(torch.abs(pred - y)) +
                             0.1 * torch.mean((pred - y) ** 2)).item()
        val_loss /= len(val_loader)

        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_loss)
        new_lr = optimizer.param_groups[0]['lr']
        lr_msg = f"  → LR dropped to {new_lr:.2e}" if new_lr < current_lr else ""
        print(f"Epoch {epoch+1:02d} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | "
              f"LR: {current_lr:.2e}{lr_msg}")

        if val_loss < best_loss:
            best_loss = val_loss
            wait      = 0
            torch.save({"model": model.state_dict()}, best_path)
            print(f"  ✓ Saved checkpoint (val_loss={val_loss:.4f})")
        else:
            wait += 1
            if wait >= patience:
                print("  Early stopping triggered")
                break

    state = torch.load(best_path, map_location=device, weights_only=True)["model"]
    model.load_state_dict(state)
    return model

In [14]:
def evaluate_model(model, loader, mean, std):
    model.eval()
    preds, trues, masks = [], [], []

    with torch.no_grad():
        for x, m, y in loader:
            x, m = x.to(device), m.to(device)
            pred, _, _ = model(x, m)
            preds.append(pred.cpu().numpy())
            trues.append(y.numpy())
            masks.append(m[:, -1, :].cpu().numpy())

    preds = np.concatenate(preds)   # (N, F)
    trues = np.concatenate(trues)   # (N, F)
    masks = np.concatenate(masks)   # (N, F)

    # --- Normalized metrics ---
    mae_norm  = mean_absolute_error(trues.ravel(), preds.ravel())
    rmse_norm = np.sqrt(mean_squared_error(trues.ravel(), preds.ravel()))

    # --- Denormalize ---
    preds_real = preds * std + mean
    trues_real = trues * std + mean

    mae_real  = mean_absolute_error(trues_real.ravel(), preds_real.ravel())
    rmse_real = np.sqrt(mean_squared_error(trues_real.ravel(), preds_real.ravel()))

    # FIXED R2 (only change)
    r2_real = r2_score(trues_real.ravel(), preds_real.ravel())

    # MAPE (unchanged)
    valid = (np.abs(trues_real) > 1e-3) & (masks == 1)
    mape  = np.mean(np.abs((trues_real[valid] - preds_real[valid]) /
                            trues_real[valid])) * 100

    print("\n==============================")
    print("Normalized Results")
    print("==============================")
    print(f"MAE  : {mae_norm:.4f}")
    print(f"RMSE : {rmse_norm:.4f}")
    print("\n==============================")
    print("Denormalized Results (Real Scale)")
    print("==============================")
    print(f"MAE  : {mae_real:.4f}")
    print(f"RMSE : {rmse_real:.4f}")
    print(f"R2   : {r2_real:.4f}")
    print(f"MAPE : {mape:.2f}%")

    return mae_real, rmse_real, r2_real, mape

In [15]:
def print_gpu_memory(label=""):
    if torch.cuda.is_available():
        alloc    = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved()  / 1e9
        print(f"[{label}] GPU: {alloc:.2f}GB allocated, {reserved:.2f}GB reserved")

In [16]:
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


def train_single_dataset(percent, model_name="pemsbay_biconvlstm"):
    print(f"\n{'='*30}\nSTARTING EXPERIMENT: {percent}% MISSING\n{'='*30}")

    raw = pd.read_csv(files[percent])
    raw = raw.apply(pd.to_numeric, errors='coerce')

    mask = (~raw.isna()).astype(np.float32).values
    data = raw.values.astype(np.float32)

    split       = int(0.8 * len(data))
    warm_end    = min(200, split)
    col_medians = np.nanmedian(data[:split], axis=0)
    col_medians = np.where(np.isnan(col_medians), 0.0, col_medians)

    for col in range(data.shape[1]):
        nan_rows = np.isnan(data[:warm_end, col])
        if nan_rows.any():
            data[:warm_end, col][nan_rows] = col_medians[col]

    mask[:warm_end, :] = np.where(
        np.isnan(data[:warm_end, :]), mask[:warm_end, :], 1.0
    )

    train_data, val_data = data[:split], data[split:]
    train_mask, val_mask = mask[:split], mask[split:]

    obs_train = train_data.copy()
    obs_train[train_mask == 0] = np.nan
    mean = np.nanmean(obs_train, axis=0, keepdims=True)
    std  = np.nanstd(obs_train,  axis=0, keepdims=True)
    mean = np.where(np.isnan(mean), 0.0, mean)
    std  = np.where(np.isnan(std) | (std < 1e-8), 1.0, std)

    print(f"--- Statistics for {percent}% Missing ---")
    print(f"Mean (first 5 sensors): {mean[0, :5]}")
    print(f"Std  (first 5 sensors): {std[0, :5]}")

    train_norm = np.where(train_mask == 1, (train_data - mean) / std, 0.0)
    val_norm   = np.where(val_mask   == 1, (val_data   - mean) / std, 0.0)

    SEQ_LEN = 10
    X_tr, M_tr, Y_tr = create_sequences(train_norm, train_mask, SEQ_LEN)
    X_vl, M_vl, Y_vl = create_sequences(val_norm,   val_mask,   SEQ_LEN)

    use_pin = device.type == "cuda"
    def make_tensor(arr):
        t = torch.tensor(arr, dtype=torch.float32)
        return t.pin_memory() if use_pin else t

    train_loader = DataLoader(
        TensorDataset(make_tensor(X_tr), make_tensor(M_tr), make_tensor(Y_tr)),
        batch_size=64, shuffle=True,  num_workers=0, pin_memory=False
    )
    val_loader = DataLoader(
        TensorDataset(make_tensor(X_vl), make_tensor(M_vl), make_tensor(Y_vl)),
        batch_size=64, shuffle=False, num_workers=0, pin_memory=False
    )

    input_dim = X_tr.shape[2]
    set_seed(42)                                                   # seed before model
    model = IConvBiLSTM(input_dim=input_dim, hidden_dim=64).to(device)
    print(f"Using device: {device}")
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

    model = train_model(model, train_loader, val_loader, percent)

    save_path = f"/kaggle/working/{model_name}_{percent}.pt"
    torch.save({"model": model.state_dict(), "mean": mean, "std": std}, save_path)
    print(f"Saved: {save_path}  ({os.path.getsize(save_path)/1e6:.1f} MB)")

    mae, rmse, r2, mape = evaluate_model(model, val_loader, mean, std)

    del model, train_loader, val_loader
    del X_tr, M_tr, Y_tr, X_vl, M_vl, Y_vl
    del train_data, val_data, train_norm, val_norm
    del train_mask, val_mask, data, mask
    gc.collect()
    torch.cuda.empty_cache()
    print_gpu_memory(f"After {percent}% cleanup")

    return mae, rmse, r2, mape

In [17]:
# import gc

# def clear_memory():
#     gc.collect()
#     torch.cuda.empty_cache()

# # ↓↓↓ ONLY CHANGE THIS ↓↓↓
# RUN_PERCENT = 10
# # ↑↑↑ ONLY CHANGE THIS ↑↑↑

# results = {}

# print(f"\n{'='*10} {RUN_PERCENT}% Missing {'='*10}")
# results[RUN_PERCENT] = train_single_dataset(RUN_PERCENT, model_name="pemsbay_IConvBilstm")
# mae, rmse, r2, mape = results[RUN_PERCENT]

# print(f"\nFINAL RESULT {RUN_PERCENT}%")
# print(f"MAE  : {mae:.4f}")
# print(f"RMSE : {rmse:.4f}")
# print(f"R2   : {r2:.4f}")
# print(f"MAPE : {mape:.2f}%")

# clear_memory()

In [18]:
import gc

def clear_memory():
    gc.collect()
    torch.cuda.empty_cache()

# ↓↓↓ ONLY CHANGE THIS ↓↓↓
RUN_PERCENT = 20
# ↑↑↑ ONLY CHANGE THIS ↑↑↑

results = {}

print(f"\n{'='*10} {RUN_PERCENT}% Missing {'='*10}")
results[RUN_PERCENT] = train_single_dataset(RUN_PERCENT, model_name="pemsbay_IConvBilstm")
mae, rmse, r2, mape = results[RUN_PERCENT]

print(f"\nFINAL RESULT {RUN_PERCENT}%")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2   : {r2:.4f}")
print(f"MAPE : {mape:.2f}%")

clear_memory()


========== 20% Missing ==========

STARTING EXPERIMENT: 20% MISSING


/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1233: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,


--- Statistics for 20% Missing ---
Mean (first 5 sensors): [ 0.       67.52889  59.069084 59.136852 62.202835]
Std  (first 5 sensors): [ 1.         8.308973  13.205921  11.783147   8.5928335]
Using device: cuda
Parameters: 467,204


Epoch 1/70:   0%|          | 0/652 [00:00<?, ?it/s]

Epoch 01 | Train: 0.2877 | Val: 0.2541 | LR: 1.00e-03
  ✓ Saved checkpoint (val_loss=0.2541)


...

Epoch 56 | Train: 0.2542 | Val: 0.2474 | LR: 1.56e-05


Epoch 57/70:   0%|          | 0/652 [00:00<?, ?it/s]

Epoch 57 | Train: 0.2541 | Val: 0.2474 | LR: 1.56e-05


Epoch 58/70:   0%|          | 0/652 [00:00<?, ?it/s]

Epoch 58 | Train: 0.2542 | Val: 0.2474 | LR: 1.56e-05
  Early stopping triggered
Saved: /kaggle/working/pemsbay_IConvBilstm_20.pt  (1.9 MB)

Normalized Results
MAE  : 0.2228
RMSE : 0.4958

Denormalized Results (Real Scale)
MAE  : 1.7905
RMSE : 4.3082
R2   : 0.7884
MAPE : 3.09%
[After 20% cleanup] GPU: 0.00GB allocated, 0.00GB reserved

FINAL RESULT 20%
MAE  : 1.7905
RMSE : 4.3082
R2   : 0.7884
MAPE : 3.09%
